In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("../data/WineQT.csv")

df.head()

In [ ]:
print("Dimensões da base:")
print(df.shape)

print("\nColunas:")
print(df.columns.tolist())

print("\nTipos de dados:")
print(df.dtypes)

In [ ]:
df.isnull().sum()

In [ ]:
df['quality_binary'] = (df['quality'] >= 7).astype(int)

df[['quality', 'quality_binary']].head(10)

In [ ]:
print(df['quality_binary'].value_counts())

print("\nPercentual das classes:")
print(df['quality_binary'].value_counts(normalize=True) * 100)

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(x='quality_binary', data=df)

plt.title('Distribuição das Classes de Qualidade')
plt.xlabel('Classe')
plt.ylabel('Quantidade de Vinhos')

plt.show()

In [ ]:
df.describe().T

In [ ]:
plt.figure(figsize=(14, 8))

df.drop(columns=['quality', 'quality_binary', 'Id']).boxplot(rot=45)

plt.title('Distribuição e possíveis outliers das variáveis químicas')
plt.ylabel('Valores')
plt.tight_layout()
plt.show()

A análise da matriz de correlação mostrou que alcohol apresentou a maior correlação positiva com a variável de classificação binária (quality_binary), com coeficiente de aproximadamente 0,40. A variável volatile acidity apresentou correlação negativa de aproximadamente -0,30, enquanto citric acid e sulphates apresentaram correlações positivas de 0,25 e 0,21, respectivamente. As demais variáveis apresentaram correlações mais fracas com a variável alvo. Esses resultados indicam possíveis associações entre as características químicas e a qualidade dos vinhos, mas não permitem estabelecer relações de causalidade.


In [ ]:
plt.figure(figsize=(12, 8))

correlacao = df.drop(columns=['Id']).corr()

sns.heatmap(
    correlacao,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0
)

plt.title('Matriz de Correlação das Variáveis')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

df = pd.read_csv("../data/WineQT.csv")

df['quality_binary'] = (df['quality'] >= 7).astype(int)

print("Base carregada!")
print("Formato:", df.shape)
print(df.columns.tolist())

In [ ]:
X = df[['alcohol', 'citric acid', 'pH', 'density', 'sulphates']]
y = df['quality_binary']

print("Formato de X:", X.shape)
print("Formato de y:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Padronização concluída!")
print("Formato X_train_scaled:", X_train_scaled.shape)
print("Formato X_test_scaled:", X_test_scaled.shape)

In [ ]:
scaler.fit_transform(X_train)

In [ ]:
scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

modelo_logistico = LogisticRegression(
    class_weight='balanced',
    random_state=42,
    max_iter=1000
)

modelo_logistico.fit(X_train_scaled, y_train)

print("Modelo de Regressão Logística treinado com sucesso!")

In [ ]:
y_pred = modelo_logistico.predict(X_test_scaled)
y_prob = modelo_logistico.predict_proba(X_test_scaled)[:, 1]

print("Previsões realizadas com sucesso!")
print("Quantidade de previsões:", len(y_pred))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

Avaliação da Regressão Logística

A Regressão Logística apresentou uma acurácia de 78,17%. Entretanto, devido ao desbalanceamento das classes, outras métricas foram consideradas para uma avaliação mais adequada do modelo. A precisão foi de 35,48%, enquanto o recall atingiu 68,75%, indicando que o modelo conseguiu identificar aproximadamente 69% dos vinhos pertencentes à classe de maior qualidade. O F1-score foi de 46,81%, refletindo o equilíbrio entre precisão e recall. O modelo apresentou ROC-AUC de 80,31%, indicando uma boa capacidade de discriminação entre as duas classes. De modo geral, os resultados demonstram que a Regressão Logística possui capacidade relevante para distinguir vinhos de maior qualidade, embora ainda apresente limitações na precisão das classificações positivas.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Qualidade < 7', 'Qualidade >= 7']
)

disp.plot()
plt.title('Matriz de Confusão - Regressão Logística')
plt.show()

Matriz de Confusão

A matriz de confusão permite analisar detalhadamente o desempenho da Regressão Logística. O modelo classificou corretamente 157 vinhos da classe de qualidade inferior a 7 e 22 vinhos da classe de qualidade igual ou superior a 7. Foram observados 40 falsos positivos, nos quais vinhos com qualidade inferior a 7 foram classificados como de maior qualidade, e 10 falsos negativos, correspondentes a vinhos de qualidade igual ou superior a 7 classificados incorretamente como pertencentes à classe inferior.

A quantidade de falsos negativos é relativamente baixa, resultando em um recall de 68,75% para a classe de maior qualidade. Entretanto, o número de falsos positivos contribuiu para a redução da precisão do modelo, que foi de 35,48%.

In [ ]:
coeficientes = pd.DataFrame({
    'Variável': X.columns,
    'Coeficiente': modelo_logistico.coef_[0]
})

coeficientes['Impacto_Absoluto'] = coeficientes['Coeficiente'].abs()

coeficientes = coeficientes.sort_values(
    by='Impacto_Absoluto',
    ascending=False
)

coeficientes

Interpretação dos coeficientes da Regressão Logística

A análise dos coeficientes do modelo indica que alcohol apresentou a maior influência na classificação, com coeficiente positivo de aproximadamente 1,22, sugerindo uma associação entre maiores teores alcoólicos e maior probabilidade de o vinho pertencer à classe de qualidade igual ou superior a 7. citric acid também apresentou influência positiva relevante (0,69), seguido por sulphates (0,56).

Entre as cinco variáveis utilizadas, pH (-0,25) e density (-0,18) apresentaram coeficientes negativos. No modelo, valores maiores dessas variáveis estão associados a uma menor probabilidade de classificação na classe de maior qualidade.

Como as variáveis foram padronizadas antes do treinamento, os coeficientes podem ser comparados em termos de magnitude. Entretanto, os resultados representam associações identificadas pelo modelo e não devem ser interpretados como relações causais.

## 5. Avaliação e interpretação da Regressão Logística

A Regressão Logística foi desenvolvida com o objetivo de classificar os vinhos em duas categorias: qualidade inferior a 7 (classe 0) e qualidade igual ou superior a 7 (classe 1).

O conjunto de dados foi dividido em 80% para treinamento e 20% para teste. As variáveis numéricas foram padronizadas antes do treinamento, garantindo que as diferentes escalas das características químicas não interferissem na modelagem.

O modelo apresentou Accuracy de 78,17%, Precision de 35,48%, Recall de 68,75%, F1-score de 46,81% e ROC-AUC de 80,31%.

Considerando o desbalanceamento das classes, a análise não deve ser baseada apenas na acurácia. O Recall de 68,75% indica que o modelo conseguiu identificar uma parcela relevante dos vinhos classificados como de maior qualidade. Além disso, o ROC-AUC de 80,31% demonstra uma boa capacidade de discriminação entre as duas classes.

A análise dos coeficientes indicou que `alcohol` apresentou a maior influência positiva na classificação, enquanto `sulphates` e `citric acid` também apresentaram associações positivas relevantes. Entre as variáveis utilizadas, `pH` e `density` apresentaram coeficientes negativos.

Apesar do desempenho satisfatório em termos de discriminação, o modelo apresentou baixa Precision, indicando uma quantidade considerável de falsos positivos. Dessa forma, a Regressão Logística apresenta potencial para a classificação da qualidade dos vinhos, mas pode ser aprimorada por meio de outros modelos e estratégias de modelagem.

## 6. Conclusão do primeiro modelo

A primeira etapa da modelagem permitiu desenvolver uma pipeline completa de análise e classificação da qualidade dos vinhos utilizando Regressão Logística.

A análise exploratória identificou um forte desbalanceamento entre as classes, com 86,09% dos vinhos classificados como qualidade inferior a 7 e 13,91% como qualidade igual ou superior a 7. Também foram identificados possíveis valores extremos em algumas variáveis, principalmente em `total sulfur dioxide` e `free sulfur dioxide`.

As análises de correlação e os coeficientes do modelo indicaram que variáveis como `alcohol`, `sulphates`, `citric acid` e `volatile acidity` possuem maior associação com a classificação da qualidade.

A Regressão Logística apresentou ROC-AUC de 80,31%, demonstrando boa capacidade de distinguir as classes. Entretanto, a Precision de 35,48% evidencia que ainda existem oportunidades de melhoria na identificação dos vinhos de maior qualidade.

Como próxima etapa do projeto, um segundo algoritmo de classificação poderá ser desenvolvido e comparado à Regressão Logística, permitindo avaliar qual abordagem apresenta melhor desempenho para o problema proposto.

In [ ]:
print("===== RESUMO DO MODELO =====")
print(f"Registros utilizados: {len(df)}")
print(f"Variáveis preditoras: {X.shape[1]}")
print(f"Treinamento: {len(X_train)} registros")
print(f"Teste: {len(X_test)} registros")
print()
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")